# 01.3 Reproducibility as an Engineering Contract

> **Prerequisites:** 01.1 (the late-payment model, precision@k, the noise band) ·
> 01.2 (the data universe, grain, content hashing)
> **What you'll learn:**
> - Measure how much a reported metric moves when nothing but the split draw changes
> - Audit *which* `random_state` in a pipeline actually affects the answer, instead of assuming
> - Make a training run byte-reproducible and prove it with a digest rather than a claim
> - Write a run manifest that makes "which model was that?" a lookup instead of an argument
> - Run reproducibility as a CI check that localizes what moved between two runs
> **Level:** Beginner · **Series:** 01 The ML Landscape & Project Lifecycle

> ⚡ **Monday 2026-03-09, 11:20** — dunning queue quality has dropped and the team decides to
> roll back to January's model. They check out the January commit, rerun training, and get a
> model that does not match January's logged numbers. Two hours into an active incident, nobody
> can say whether the old model was better or whether they are looking at a different model
> entirely. The cause: the rollback target was a script, and the script was never a function of
> anything anyone recorded.

## Concept
### Plain-English Explanation

01.1 ended with a decision: keep the rule, hold the model in shadow, revisit when the shadow
model's value clears the noise band for four consecutive weeks. That decision rests entirely on
comparing numbers produced at different times by different people. If those numbers cannot be
regenerated, the decision procedure is theatre — not because anyone is dishonest, but because
"the model out-scored the rule on precision" is a claim about an experiment, and an experiment
that cannot be repeated is an anecdote.

Reproducibility is usually taught as hygiene: set your seeds, pin your versions, be tidy. That
framing is why it gets skipped. The engineering framing is that a training run is a **function**,
its inputs are code, data and configuration, and reproducibility means having actually recorded
the inputs. When a rollback is needed at 11:20 on a Monday, the question "which model was
running in January?" must have an answer that is a lookup rather than a reconstruction.

The uncomfortable finding below is that the usual ritual — sprinkling `random_state=42` across
every constructor — pins almost nothing on this pipeline, while the one source of variation
nobody was tracking moves the headline metric by twice the size of the improvement that
justified the project.

### Technical Explanation

A training run maps `(code, data, config) → (artifact, metrics)`. Reproducibility is the claim
that re-applying the same three inputs yields the same two outputs. It fails when any input is
not actually recorded, and the three fail in different ways.

**Code** is the input teams do record, because version control exists. It is also the least
common cause of irreproducibility, which is why rebuilding from a git SHA feels like it should
work and does not.

**Data** is usually a live query — `WHERE issue_date < today` — so the same script run in
January and in June trains on different rows. PayFlow's training pool grows from 159,821 to
187,111 invoices across those five months. The remedy is a **content hash**: the same technique
01.2 used to fingerprint the exports, applied to the exact slice a run consumed, so "which data"
becomes a recorded fact rather than a memory.

**Config** includes the seed, and here is where intuition misleads. Randomness does not enter a
pipeline diffusely; it enters at enumerable points, and each one either affects the output or
does not. Setting `random_state` on `LogisticRegression` with the lbfgs solver changes nothing,
because lbfgs is deterministic — the parameter exists for solvers that need it. Meanwhile the
train/test split is drawn at random, and *that* moves the metric across a range of 0.0378,
which is 2.0 times the 0.0194 improvement 01.1 measured. ⭐ **CRITICAL CONCEPT** — you cannot
seed your way to reproducibility by decorating constructors. You have to enumerate where
randomness enters, test each point, and pin the ones that matter.

Two distinctions do real work later. **Identity** is stricter than **equivalence**: two models
trained on slightly different data can behave almost identically — differing on 0.6% of queue
slots — while being entirely different artifacts with different digests. And a **manifest** is
not a log. A log records that something happened; a manifest records enough to make it happen
again.

### Mental Model

A training run is a function, and its inputs are code, data and config. Reproducibility is not
tidiness — it is having written down all three arguments, so that "which model was that?" is a
lookup rather than an archaeology project conducted during an incident.

## How It Works

```text
   INPUTS (must all be recorded)                     OUTPUTS (must both be addressable)
   ---------------------------------                 ---------------------------------
   code    git SHA + library versions  ----+
                                           |
   data    content hash of the exact  -----+---->  [ training run ]  ---->  artifact  (hash)
           slice consumed, row count       |                               metrics   (value)
                                           |
   config  seed, split id, features,  -----+
           hyperparameters

   Where randomness actually enters:              Effect on the answer:
     train_test_split(random_state=)      ->  SEED-SENSITIVE   (spread 0.0378 on precision@k)
     DataFrame.sample(random_state=)      ->  SEED-SENSITIVE
     SGDClassifier(random_state=)         ->  SEED-SENSITIVE   (stochastic solver)
     LogisticRegression(lbfgs, rs=)       ->  no effect at all (deterministic solver)
```

Read the right-hand column as the notebook's argument. Three of the four knobs labelled
`random_state` change the answer and one does not, and which is which cannot be guessed from the
name of the parameter — it follows from whether the underlying algorithm consumes randomness.
The lbfgs solver optimizes a convex objective by a deterministic procedure; given the same rows
in the same order it returns the same coefficients, so its `random_state` is inert. `SGDClassifier`
shuffles its data every epoch, so its seed is load-bearing. And `train_test_split` decides which
rows the model ever sees, which is the largest lever of all.

That last point resolves something 01.1 left implicit. The bootstrap there resampled the
*training* pool while scoring a fixed evaluation window, and reported a reassuringly tiny
standard deviation — the fitted coefficients barely move when you jiggle 160,000 training rows.
What it never measured is what happens when the *evaluation* rows change. Those are two
different quantities that both get called variance, and the cell below measures them side by
side: training stability comes in at a spread of 0.0078, evaluation stability at 0.0378, a
ratio of 4.9x. The larger one is the one nobody was pinning.

The remedy is a **manifest**: a record of every input and a content address for every output.
Given two manifests you can localize what moved — code, data or config — which turns "the
numbers don't match" from a mystery into a diff.

## Hands-On Build
### Stage A — from scratch

The claim to establish first is that the variation is real and large. No framework: draw the
evaluation split by hand with an explicit random generator, refit, and record what the headline
metric does. Twelve draws, nothing else changed — same code, same data, same features, same
model.

In [1]:
# Load the committed lab module for 01.3; it reuses 01.1's dataset builder rather than
# duplicating it, which is the same discipline this notebook argues for.
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd

LAB = Path.cwd() / "_lab" / "lab_01.3_reproducibility.py"
spec = importlib.util.spec_from_file_location("lab_01_3", LAB)
lab = importlib.util.module_from_spec(spec)
sys.modules["lab_01_3"] = lab
spec.loader.exec_module(lab)

df, _ = lab.lab11.build_dataset()          # the 01.1 modelling table, unchanged
pool = lab.lab11.windows(df)["train"]      # invoices issued before 2025-01-01
print(f"training pool: {len(pool):,} invoices\n")

# Stage A: draw the split by hand so the source of randomness is explicit and visible.
def manual_split(frame: pd.DataFrame, n_eval: int, seed: int):
    rng = np.random.default_rng(seed)                 # the ONLY randomness in this cell
    order = rng.permutation(len(frame))
    return frame.iloc[order[n_eval:]], frame.iloc[order[:n_eval]]

results = {}
for s in range(6):
    train, test = manual_split(pool, lab.EVAL_ROWS, seed=s)
    scores = lab.lab11.fit_score(train, test, seed=lab.SEED)
    k = int(lab.lab11.dunning_rule(test).sum())
    results[s] = lab.lab11.precision_recall_at_k(
        test["late"].to_numpy(), lab.lab11.topk_flag(scores, k))["precision"]

series = pd.Series(results, name="precision@k")
print(series.round(4).to_string())
print(f"\nspread across 6 hand-drawn splits: {series.max() - series.min():.4f}")

training pool: 159,821 invoices



0    0.4326
1    0.4617
2    0.4367
3    0.4229
4    0.4464
5    0.4470

spread across 6 hand-drawn splits: 0.0388


Six draws, and the metric wanders. The only thing that changed between runs is which rows landed
in the evaluation sample — the model class, the features, the hyperparameters and the training
procedure were identical throughout. Whatever number a single run of this script reports is one
sample from a distribution, and reporting it without the distribution is reporting a coin flip
as a measurement.

### Stage B — idiomatic

Same experiment through `train_test_split`, at twelve draws for a tighter estimate, and then the
audit that matters: which of this pipeline's `random_state` parameters actually moves the
answer. That question is answered by testing, not by reading constructor signatures.

In [2]:
runs = lab.split_lottery(pool, n_runs=12)      # sklearn train_test_split, seeds 0..11
print(runs.round(4).to_string())
spread = runs.max() - runs.min()
print(f"\nmean {runs.mean():.4f}   std {runs.std():.4f}   "
      f"min {runs.min():.4f}   max {runs.max():.4f}   spread {spread:.4f}")
print(f"01.1 claimed a model-over-rule gain of {lab.CLAIMED_GAIN:+.4f}; the split lottery "
      f"alone spans {spread / lab.CLAIMED_GAIN:.1f}x that\n")

# Which variance did 01.1 actually measure? Resampling the TRAINING set with the evaluation
# rows held fixed is a different experiment from re-drawing the evaluation rows.
lab.training_vs_evaluation_noise(pool)

0     0.4366
1     0.4476
2     0.4394
3     0.4620
4     0.4419
5     0.4323
6     0.4461
7     0.4242
8     0.4313
9     0.4282
10    0.4430
11    0.4427

mean 0.4396   std 0.0101   min 0.4242   max 0.4620   spread 0.0378
01.1 claimed a model-over-rule gain of +0.0194; the split lottery alone spans 2.0x that



  training stability   (fixed eval rows, resampled training set): std 0.0032   spread 0.0078
  evaluation stability (fixed procedure, re-drawn eval rows):     std 0.0113   spread 0.0378
  ratio of spreads: 4.9x  -> the holdout draw dominates, and it is the one nobody was pinning


The spread is 0.0378 against a claimed improvement of 0.0194. A single unseeded run of this
pipeline can honestly report anything between 0.4242 and 0.4620 with no code change whatsoever,
which means a team running the experiment once and shipping on the result is deciding by lottery
and calling it evidence. This is the mechanical justification for the noise band 01.1 insisted
on: not a statistical nicety, but the difference between a measurement and a draw.

The second block is the correction to 01.1's own method. Holding the evaluation rows fixed and
resampling the training set — which is what that notebook's bootstrap did — gives a spread of
0.0078. Re-drawing the evaluation rows instead gives 0.0378, larger by a factor of 4.9. Both
are legitimate measurements of different things, and a team that reports only the first will
systematically understate how much its number can move, because the quantity it left unmeasured
is the dominant one.

Now the audit. Four components in this pipeline accept a `random_state`; the question is which
of them the answer depends on.

In [3]:
train_fixed, test_fixed = lab.train_test_split(pool, test_size=lab.EVAL_ROWS,
                                               random_state=lab.SEED)
lab.seed_sensitivity(pool, train_fixed, test_fixed)
print()
lab.determinism_check(train_fixed, test_fixed)

  train_test_split(random_state=s)             ed32142305fc / 0870943ba184  -> SEED-SENSITIVE


  LogisticRegression(lbfgs, random_state=s)    9c657ec4f4d4 / 9c657ec4f4d4  -> seed changes NOTHING


  SGDClassifier(log_loss, random_state=s)      dd2f3c8c563a / 86d346e6ac50  -> SEED-SENSITIVE


  DataFrame.sample(random_state=s)             17ac5bffbb3c / 31c410887c6b  -> SEED-SENSITIVE

  Seeding a deterministic estimator is theatre: it looks like diligence and
  pins nothing. Randomness enters through row selection - splits, resampling,
  and stochastic solvers - so those are what a manifest has to record.



  3 repeats, identical seed/data/code -> digests ['9c657ec4f4d4', '9c657ec4f4d4', '9c657ec4f4d4']
  all identical: True


['9c657ec4f4d4', '9c657ec4f4d4', '9c657ec4f4d4']

⚠️ `LogisticRegression(random_state=42)` — the parameter set dutifully in 01.1, and in a large
share of production pipelines — changes nothing. The lbfgs solver is deterministic, so the seed
is inert, and a team that sets it and stops has pinned none of the three knobs that do matter.
`SGDClassifier` tells the opposite story with the identical parameter name, because its solver
consumes randomness. The lesson generalizes past this pipeline: seed sensitivity is a property
of the algorithm, is not documented by the presence of the parameter, and takes about four lines
to test.

With the real sources pinned — a fixed split, a fixed data slice, a deterministic estimator —
three consecutive runs produce the identical digest `9c657ec4f4d4`. That digest is the claim
this notebook is building toward: not "the pipeline is reproducible" as an assurance, but a
value that either matches or does not.

### Stage C — production

Determinism inside one session is necessary and nowhere near sufficient. The incident opens with
a rollback across two months, so what has to survive is the *record*: the manifest below captures
code, data, config and result, and content-addresses both the data consumed and the predictions
produced.

In [4]:
import json

# Inputs (code / data / config) and outputs (result) recorded as separate blocks, so a
# change to what you asked for is never confused with a change to what you got.
manifest = lab.build_manifest(train_fixed, test_fixed, lab.SEED)
print(json.dumps(manifest, indent=2))

{
  "code": {
    "python": "3.14.4",
    "numpy": "2.5.2",
    "pandas": "3.0.5",
    "sklearn": "1.9.0"
  },
  "data": {
    "train_rows": 154321,
    "train_hash": "7fc3bfc951a5",
    "eval_rows": 5500,
    "eval_hash": "ed32142305fc"
  },
  "config": {
    "seed": 42,
    "features_num": 6,
    "features_cat": 4,
    "queue_k": 1412
  },
  "result": {
    "score_hash": "9c657ec4f4d4",
    "precision_at_k": 0.439093
  }
}


Four blocks, and the split between them is the design. `code` and `data` and `config` are the
function's arguments; `result` is its output. Recording the arguments is what makes the run
repeatable; content-addressing the output is what makes the repeat *checkable*. A log line
saying "trained model, precision 0.439093" supports neither.

Note what `data` records: not a filename or a date, but `train_hash` and `train_rows` — the
fingerprint of the exact slice consumed. `_data/SPEC.md` applies the same idea one level up, to
the generated exports themselves, so the chain from raw file to reported metric is addressable
end to end.

## Evaluation

The thing being measured is the pipeline's own reproducibility, so the metric is digest equality
across repeats and the harness is `reproduce()`: rebuild the manifest from the recorded inputs
and diff it field by field. The baseline to beat is the unpinned pipeline from Stage B, whose
result varies across a 0.0378 range. A pass is an exact match on every field; anything else is a
failure whose value is that it names which input moved.

In [5]:
print(" (a) same code, same data, same config:")
lab.reproduce(manifest, train_fixed, test_fixed)

print("\n (b) the data grew under an unchanged script (the common real case):")
grown = pd.concat([train_fixed, lab.lab11.windows(df)["test_stable"].head(2_000)])
lab.reproduce(manifest, grown, test_fixed)

print("\n (c) the seed changed - does the record stay honest?")
reseeded = lab.build_manifest(train_fixed, test_fixed, lab.SEED + 1)
print(f"    fields that moved: {lab.diff_manifest(manifest, reseeded)}")

 (a) same code, same data, same config:


  REPRODUCED: manifest matches on code, data, config and result

 (b) the data grew under an unchanged script (the common real case):


  NOT REPRODUCED (4 field(s) moved):
    - data.train_hash: '7fc3bfc951a5' -> 'd7712e31f535'
    - data.train_rows: 154321 -> 156321
    - result.precision_at_k: 0.439093 -> 0.439802
    - result.score_hash: '9c657ec4f4d4' -> 'b1040ed2b5f8'

 (c) the seed changed - does the record stay honest?


    fields that moved: ['config.seed: 42 -> 43']


Case (a) is the contract holding: every field matches, so the January number can be regenerated
in March and the rollback comparison means something.

Case (b) is the incident in miniature, and the payoff of recording data rather than trusting it.
Two thousand extra invoices arrive under an unchanged script, and the check fails with the
failure *localized*: `data.train_rows` moved from 154,321 to 156,321, `data.train_hash` changed,
and consequently `result.score_hash` and `result.precision_at_k` moved too. Nobody has to
speculate about whether the code changed; the diff says it did not. Compare that to the incident
timeline, where the same underlying situation consumed two hours of an outage.

Case (c) is the honest bookkeeping the earlier audit demands. Changing the seed moves exactly one
field — `config.seed` from 42 to 43 — and no result field, because this estimator is
seed-independent. That is the correct behaviour for a manifest: it separates what you *asked
for* from what you *got*, so an inert configuration change shows up as a configuration
difference rather than being silently mistaken for a reproduced experiment.

One honest limitation. Digest equality here is within a pinned environment on one machine.
Bitwise-identical results across machines additionally depend on BLAS threading, CPU instruction
sets and library builds, which is why the manifest records library versions and why serious
setups also record thread counts. Reproducibility is a property of a stated environment, not an
absolute, and the manifest is where that environment gets stated.

## Design Patterns / Tradeoffs

**Rebuild from source versus store the artifact.** Rebuilding keeps only code in version control
and regenerates the model on demand. It is cheap to store, guarantees the build path still works,
and catches rot in the pipeline. It also fails exactly as the cold open describes: rebuilding
reproduces the *process*, not the *object*, so it is only as trustworthy as the recording of
data and config, and it is the slowest possible thing to attempt during an incident. Storing the
artifact — the serialized fitted pipeline plus its manifest, immutable and content-addressed —
makes rollback a download. Its costs are real: storage grows with every run, artifacts are
opaque binaries that silently depend on library versions (01.2's pickle-across-versions warning
applies), and an artifact with no manifest is a black box you can serve but cannot explain. Use
stored artifacts as the rollback target for anything serving traffic; use rebuild-from-source as
a scheduled audit that the recorded inputs really do regenerate the recorded outputs.

**Fixed evaluation set versus a split re-drawn each run.** A fixed, versioned evaluation set makes
runs comparable — the entire point when the decision is "is this better than last month's" — and
it is what makes a 0.0194 improvement legible at all against a lottery spread of 0.0378. Its
danger is overfitting to the holdout through repeated selection, which is real and grows with
every decision made against it. Re-drawing the split each run avoids that but destroys
comparability across runs, which is worse for a team shipping incrementally. The standard
resolution is a fixed evaluation set for run-to-run comparison plus repeated resampling for the
uncertainty estimate — measure the spread, then report against a stable set — with the holdout
rotated on a schedule and a genuinely untouched set reserved for final confirmation. Series 12
makes this canonical.

**Recommendation for PayFlow:** immutable artifacts plus manifests as the rollback path, a fixed
monthly evaluation set for comparisons, the bootstrap band from 01.1 reported beside every
claimed delta, and `reproduce()` in CI on the training path. The condition that changes it is
scale — if training grows expensive enough that storing every artifact is impractical, keep
artifacts only for runs that were promoted, and keep manifests for all of them.

## Production Scenario
### Symptoms

**Monday 2026-03-09, 11:20.** Dunning queue quality has been degrading for a week and the team
elects to roll back to the model that was running in January. There is no artifact store; the
deployment process builds the model from source, so "roll back" means checking out the January
commit and running the training job.

What the on-call sees:

- **11:20** — the rebuilt January model deploys and its scores do not match the scores logged in
  January for the same invoices. Same commit, same script, different model.
- **11:35** — someone reruns the January job a second time to check, and gets a third set of
  numbers. Nothing errored in any of the three runs.
- The pull request that approved the January model quotes a precision figure that nobody can now
  regenerate, and the improvement it claimed was +0.0194.
- The training query has no date bound: it reads every invoice with a resolved payment as of the
  moment it runs. Between January and June the pool grows from 159,821 to 187,111 rows.
- Two hours in, the incident channel is arguing about whether January was genuinely better,
  with no way to settle it, while the degraded queue is still serving.

In [6]:
lab.data_lottery(df, lab.lab11.windows(df)["test_stable"].head(lab.EVAL_ROWS))

  as-of 2025-01-01: train_rows=159,821  data_hash=b713c0265150  score_hash=bbc3210af25b


  as-of 2025-06-01: train_rows=187,111  data_hash=fcb7f4d22f5e  score_hash=f40d77ccb8df
  mean |score difference| on identical rows: 0.0018
  queue disagreement: 18 of 3,048 slots (0.6% of the two queues differ)
  precision@k 0.4462 vs 0.4475  (delta +0.0013, claimed gain in 01.1 was +0.0194)


### Diagnosis

Walking the ladder, and naming what each signal eliminated:

1. **Alert** — queue quality degradation triggers a rollback; the rollback's own validation step
   fails because rebuilt scores do not match logged scores. Candidate causes: a code change, a
   data change, a library change, or nondeterminism in the pipeline.
2. **Model-quality comparison** — the rebuilt model's precision differs from January's logged
   figure. This confirms the artifacts differ but says nothing about why, and critically does not
   yet distinguish "the model is worse" from "the model is different".
3. **Prediction logs** — re-scoring January's invoices with the rebuilt model produces different
   scores on identical inputs, so the two artifacts are different functions rather than the same
   function seeing different data at serve time. The problem is upstream, in training.
4. **Local reproduction** — running the January script twice on the same day yields two different
   results. This is decisive: the pipeline is nondeterministic, so "the January model" never named
   one object. The measured spread of 0.0378 exceeds the +0.0194 improvement that justified
   shipping, which means the original approval could not have been supported by its own evidence.
5. **Data check** — the training query has no snapshot bound and the pool has grown from 159,821
   to 187,111 rows. The cell above isolates this factor alone: holding code and split fixed and
   varying only the data snapshot, scores shift by a mean absolute 0.0018 and the two queues
   disagree on 18 of 3,048 slots. The data is a genuine input, but here a small one.
6. **Version diff** — no code change between the January commit and the rebuild, by construction.
   Library versions are unrecorded, which is a gap rather than a finding.

The decomposition matters: the split lottery contributes 0.0378 of spread and the data drift
contributes 0.0013 of precision movement, so the dominant term is the unpinned split, not the
growing dataset that the team spent its first hour discussing.

### Root Cause

The training pipeline's output is a function of code, data snapshot and split draw, and only
code was under version control. "January's model" therefore names a distribution of possible
models spanning 0.4242 to 0.4620 in precision — a range 2.0 times the improvement that
justified deploying it — so no rebuild could have matched the logged numbers, and no comparison
between the rebuild and the current model could settle which was better.

### Fix

**Mitigation now.** Stop rebuilding. Freeze the currently deployed artifact, and if queue quality
is unacceptable, fall back to the dunning rule from 01.1 — which is deterministic by
construction and needs no training run at all. Pin the comparison to a fixed list of invoice ids
so that at least the decisions made during the incident are internally consistent.

**Permanent fix.** Every training run writes a manifest of the shape shown above and stores its
artifact immutably, keyed by the artifact digest. The evaluation split stops being drawn and
becomes a versioned, fixed set of invoice ids. The training query takes a snapshot bound and the
consumed slice is content-hashed. Rollback becomes: look up the manifest, fetch the artifact by
digest, deploy. No rebuild, no lottery.

### Prevention

- **`reproduce()` in CI** on every pull request touching the training path: rebuild the last
  manifest and fail the build on any field diff. Case (b) above is exactly this check catching
  exactly this class of failure, in seconds rather than hours.
- **A manifest, not a number, in the PR template.** A precision figure with no manifest is not
  reviewable; the reviewer cannot tell a real improvement from a lucky draw.
- **Report the noise band beside every claimed delta**, per 01.1. The January approval would have
  been rejected on its own evidence had the 0.0378 spread been reported next to the +0.0194 gain.
- **Audit seed sensitivity when the pipeline changes.** The four-line test above tells you which
  knobs matter; assuming `random_state` is sufficient is what left the split unpinned while
  everyone believed the run was seeded.
- **Content-address data snapshots** the way `_data/SPEC.md` does for the raw exports, so "which
  data" is answerable months later without reconstructing a query's mood on a given Tuesday.

## Common Pitfalls

⚠️ **Setting `random_state` everywhere and believing the run is pinned.** On this pipeline the
estimator's seed changes nothing while the split's seed moves the metric by 0.0378. The tell is
that nobody has tested which is which; the fix is four lines and takes a minute.

**Rolling back to a commit instead of an artifact.** A commit reproduces the process, not the
object. Unless data and config are recorded too, rebuilding is a fresh draw from the same
distribution — which is precisely how a rollback turns into a two-hour argument.

**Training from a live query with no snapshot bound.** `WHERE issue_date < today` makes the
current date a hidden input. Bound the query and hash the slice, or the model's identity depends
on when someone happened to press run.

**Reporting a single run's metric with no spread.** One number from a stochastic procedure is a
sample, not a measurement. Any claimed improvement smaller than the run-to-run spread is
unsupported by construction.

**Comparing two runs whose evaluation sets differ.** Half the apparent movement between two
experiments is often the holdout changing underneath them. Fix the evaluation set first, then
compare.

⚠️ **Reading a digest mismatch as a behavioural difference.** The two data-snapshot models here
have completely different digests and disagree on only 0.6% of queue slots. Identity is stricter
than equivalence: a hash tells you *whether* something changed, never *how much it matters*.
Use the digest to detect, then measure the effect before reacting.

**Recording metrics but not library versions.** A pickled model can change behaviour under a
library upgrade with no code change at all, and without recorded versions there is no way to
attribute the shift.

## Interview Questions

1. **Derive this.** A team evaluates on a random holdout of *n* rows and reports precision@k.
   Explain where the run-to-run variance comes from and how it scales with *n*, then explain why
   a training-set bootstrap and a re-drawn holdout give spreads of 0.0078 and 0.0378 on the same
   pipeline. *Answer shape:* the bootstrap resamples a six-figure training pool and measures
   coefficient stability, which is high because the fit is well determined; re-drawing the
   holdout varies which 5,500 rows are scored, and noise on a proportion scales roughly as the
   inverse square root of the queue size, so a small evaluation sample dominates. They are
   different quantities and the larger one bounds what a single run can report.
2. **Design this.** Design the reproducibility contract for a team shipping a model monthly.
   *Answer shape:* immutable artifacts keyed by digest plus manifests recording code versions,
   data hash and row count, config and result; a fixed versioned evaluation set; snapshot-bounded
   training queries; `reproduce()` in CI on the training path; manifests required in PR review;
   noise band reported beside every claimed delta.
3. **Debug this.** A colleague reports a two-point precision improvement. You rerun their exact
   script and get a different number. Walk your diagnosis. *Answer shape:* rerun twice more to
   establish whether the pipeline is deterministic at all; if not, measure the spread and compare
   it to the claimed effect; if it is, diff the inputs — data hash and row count first, then
   library versions, then config; the manifest diff localizes it in one step if one exists.
4. Which parameters named `random_state` in a scikit-learn pipeline actually affect the output,
   and how would you find out? *Answer shape:* those belonging to components that consume
   randomness — splitters, resamplers, stochastic solvers, tree ensembles' bootstrap and feature
   subsampling — but not deterministic solvers like lbfgs. Do not reason from the signature; run
   the component twice with two seeds and compare digests.
5. Your pipeline is byte-reproducible on your laptop. What is still not guaranteed, and what
   would you record to close the gap? *Answer shape:* cross-machine bitwise equality depends on
   BLAS threading, CPU instruction sets and library builds; record library versions, thread
   counts and ideally a container digest, and treat reproducibility as a property of a stated
   environment.
6. Two model artifacts have different digests but disagree on 0.6% of decisions. Are they the
   same model? *Answer shape:* not the same artifact, effectively equivalent in behaviour. The
   digest answers identity and is the right trigger for investigation; the disagreement rate
   answers impact and is the right input to a decision. Conflating them produces either false
   alarms or missed drift.
7. Why is a manifest different from a log line, and what specifically must it contain? *Answer
   shape:* a log records that something happened; a manifest records enough to make it happen
   again — every input (code versions, data hash and row count, config and seed) and a content
   address for every output, with inputs and outputs kept in separate blocks so an inert config
   change is visibly distinct from a changed result.

## Key Takeaways

- A training run is a function of code, data and config; recording only code is why rebuilding
  from a commit does not reproduce a model.
- Test which `random_state` matters rather than assuming — the estimator's seed here changes
  nothing while the split's seed moves precision by 0.0378.
- Evaluation-set noise and training-set noise are different quantities: 0.0078 spread from
  resampling the training set against 0.0378 from re-drawing the holdout, a factor of 4.9.
- Any claimed improvement smaller than the run-to-run spread is unsupported: the +0.0194 gain sat
  inside a lottery spanning 0.4242 to 0.4620.
- Content-hash the exact data slice a run consumed, so "which data" is a recorded fact rather
  than a reconstruction performed during an incident.
- Keep manifest inputs and outputs in separate blocks, so an inert config change reads as a
  config difference and never as a reproduced experiment.
- Roll back to an immutable artifact, not to a commit; rebuild-from-source belongs in a scheduled
  audit, not on the incident path.
- Digest inequality detects change but does not measure it — two artifacts here differ completely
  while disagreeing on 0.6% of decisions.

## Related

**Backward**

- **01.1 Rules or Learning?** — supplies the model, the precision@k harness and the +0.0194 claim
  whose support this notebook re-examines; its bootstrap measured training stability, and the
  larger evaluation-stability term it did not measure is quantified here.
- **01.2 First Contact with the PayFlow Data Universe** — introduced content hashing as a data
  contract; the `train_hash` field here is that idea applied to a training slice.

**Forward**

- **01.4 The Taxonomy of Learning Problems** — where the shape of each problem type determines
  what a run's config block even has to contain.
- **01.6 The ML Project Lifecycle** — places manifests and artifact stores in the lifecycle,
  between evaluation and deployment.
- **12.1 Model Evaluation & Validation** — canonical home for fixed versus rotated holdouts,
  cross-validation, and the multiple-comparisons problem that repeated selection against one
  evaluation set creates.
- **20.1 Ensembles & Gradient Boosting** — tree ensembles are genuinely seed-sensitive through
  bootstrap and feature subsampling, so the audit built here becomes load-bearing there.
- **34.1 ML in Production** — the artifact registry, model versioning and CI gating that this
  notebook's manifest is the minimal form of.